In [44]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# random seed for reproducibility
import tensorflow as tf
import numpy as np
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


In [45]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/ML/Dummy data set/Per_Trail_EEG-2.xlsx")

# Convert ASD/TD to 1/0
df['Label_Class'] = df['DataSet'].apply(lambda x: 1 if pd.notna(x) and 'ASD' in x else (0 if pd.notna(x) and 'TD' in x else None))

# EEG Feature Columns
features = [
    'AF7_Alpha', 'AF7_Beta', 'AF7_Theta',
    'AF8_Alpha', 'AF8_Beta', 'AF8_Theta',
    'Theta/Alpha_AF7', 'Alpha/Beta_AF7',
    'Theta/Alpha_AF8', 'Alpha/Beta_AF8'
]

X = df[features]
y = df['Label_Class']# Filter for ASD and TD only
df = df[df['DataSet'].isin(['ASD', 'TD'])]
df['target'] = df['DataSet'].apply(lambda x: 1 if 'ASD' in x else 0)


In [47]:

#  Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20, random_state=42)


In [48]:
from tensorflow.keras.optimizers import Adam

# Learning rates to test
learning_rates = [0.0001, 0.001, 0.01]

results = []
best_model = None
best_score = 0
best_lr = None

for lr in learning_rates:
    print(f"\n🔁 Training with learning_rate = {lr}")

    # Build model with current learning rate
    optimizer = Adam(learning_rate=lr)

    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model.fit(X_train, y_train, epochs=30, batch_size=16, validation_split=0.10, verbose=0, callbacks=[early_stop])

    score = model.evaluate(X_test, y_test, verbose=0)[1]
    results.append((lr, score))

    if score > best_score:
        best_score = score
        best_model = model
        best_lr = lr

# Show results
for lr, acc in results:
    print(f"📊 Learning Rate: {lr} → Test Accuracy: {acc:.2f}")

print("\n✅ Best Learning Rate:", best_lr)
print("✅ Best Accuracy: {:.2f}".format(best_score))




🔁 Training with learning_rate = 0.0001

🔁 Training with learning_rate = 0.001

🔁 Training with learning_rate = 0.01
📊 Learning Rate: 0.0001 → Test Accuracy: 0.42
📊 Learning Rate: 0.001 → Test Accuracy: 0.58
📊 Learning Rate: 0.01 → Test Accuracy: 0.67

✅ Best Learning Rate: 0.01
✅ Best Accuracy: 0.67


In [49]:
# Grid Search-style Hyperparameter Tuning for Neural Network
param_grid = {
    "hidden_units": [16, 32],
    "activation": ['relu', 'tanh'],
    "batch_size": [16, 32],
    "epochs": [30, 50]
}

results = []
best_model = None
best_score = 0
best_params = {}

def build_nn(hidden_units=32, activation='relu', input_dim=10, learning_rate=0.001):
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Input
    from tensorflow.keras.optimizers import Adam

    optimizer = Adam(learning_rate=learning_rate)

    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(hidden_units, activation=activation))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model


#  Search through all combinations
for hu in param_grid["hidden_units"]:
    for act in param_grid["activation"]:
        for bs in param_grid["batch_size"]:
            for ep in param_grid["epochs"]:
                print(f"Training with: hidden_units={hu}, activation={act}, batch_size={bs}, epochs={ep}")
                model = build_nn(hidden_units=hu, activation=act, input_dim=X_train.shape[1], learning_rate=best_lr)
                early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                model.fit(X_train, y_train, epochs=ep, batch_size=bs, validation_split=0.2, verbose=0, callbacks=[early_stop])
                score = model.evaluate(X_test, y_test, verbose=0)[1]  # accuracy
                if score > best_score:
                    best_score = score
                    best_model = model
                    best_params = {
                        "hidden_units": hu,
                        "activation": act,
                        "batch_size": bs,
                        "epochs": ep
                    }

#  Final Output Like SVM
print("✅ Best Hyperparameters:", best_params)
print("✅ Best Test Accuracy: {:.2f}".format(best_score))


Training with: hidden_units=16, activation=relu, batch_size=16, epochs=30
Training with: hidden_units=16, activation=relu, batch_size=16, epochs=50
Training with: hidden_units=16, activation=relu, batch_size=32, epochs=30
Training with: hidden_units=16, activation=relu, batch_size=32, epochs=50
Training with: hidden_units=16, activation=tanh, batch_size=16, epochs=30
Training with: hidden_units=16, activation=tanh, batch_size=16, epochs=50
Training with: hidden_units=16, activation=tanh, batch_size=32, epochs=30
Training with: hidden_units=16, activation=tanh, batch_size=32, epochs=50
Training with: hidden_units=32, activation=relu, batch_size=16, epochs=30
Training with: hidden_units=32, activation=relu, batch_size=16, epochs=50
Training with: hidden_units=32, activation=relu, batch_size=32, epochs=30
Training with: hidden_units=32, activation=relu, batch_size=32, epochs=50
Training with: hidden_units=32, activation=tanh, batch_size=16, epochs=30
Training with: hidden_units=32, activa

In [50]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

#  Prepare full data (as before)
X = df.drop(['Label', 'Label_Class', 'DataSet', 'Trial'], axis=1)
X = X.astype('float32')
y = df['Label_Class']  # ASD vs TD

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ✅ Step 2: Build and train the model using best hyperparameters
model = build_nn(
    hidden_units=best_params['hidden_units'],
    activation=best_params['activation'],
    input_dim=X_train.shape[1],
    learning_rate=0.001
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=best_params['epochs'],
          batch_size=best_params['batch_size'],
          validation_split=0.1, verbose=0, callbacks=[early_stop])

# ✅ Step 3: Add task info back to test set for evaluation
X_test_copy = X_test.copy()
X_test_copy['true_label'] = y_test.values
X_test_copy['predicted_label'] = (model.predict(X_test) > 0.5).astype("int32")
X_test_copy['task'] = df.loc[X_test.index, 'Label']  # Add back task info (BM, BS, NM)

# ✅ Step 4: Loop through tasks and print task-wise performance
for task in sorted(X_test_copy['task'].unique()):
    task_data = X_test_copy[X_test_copy['task'] == task]
    print(f"\n📌 Task {task} — Classification Report (Single Trained Model)")
    print(classification_report(
        task_data['true_label'], task_data['predicted_label'],
        labels=[0, 1],
        target_names=["TD", "ASD"],
        zero_division=0
    ))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step

📌 Task 0 — Classification Report (Single Trained Model)
              precision    recall  f1-score   support

          TD       0.50      1.00      0.67         1
         ASD       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2


📌 Task 1 — Classification Report (Single Trained Model)
              precision    recall  f1-score   support

          TD       0.50      1.00      0.67         1
         ASD       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3


📌 Task 2 — Classification Report (Single Trained Model)
              precision    recall  f1-score   support

          TD       1.00      0.50      0.67         2
         ASD       0.50     